# End-to-End NDPI Pipeline Demo (In-Process)

This notebook runs the full preprocessing + RF-DETR training + annotation pipeline
by importing functions directly from src/ (no subprocess calls).
Update the paths in the next cell to match your data layout.

Steps:
1. Read NDPI+NDPA pairs and generate H5 tiles
2. Postprocess tiles (focus stack + rankings)
3. Split into train/val/test
4. Export COCO (single class, filtered boxes)
5. Train RF-DETR
6. Run annotator on a new slide

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

# ---- Update these paths ----
repo_root = Path('/home/ats16/dsci435/Smithsonian_fossil_Sp26')
input_ndpi_dir = Path('/path/to/ndpi_with_ndpa')  # contains .ndpi + matching .ndpi.ndpa
annotation_map_csv = Path('/path/to/annotation_categories.csv')

# Output roots
h5_dir = repo_root / 'output' / 'tiles_h5'
splits_dir = repo_root / 'output' / 'splits'
coco_dir = repo_root / 'output' / 'coco_export'
rfdetr_out = repo_root / 'output' / 'rfdetr_run'
annotator_out = repo_root / 'output' / 'annotator'

# New slide to annotate (not part of train/val/test)
annotate_ndpi_path = Path('/path/to/new_slide.ndpi')

# Hyperparameters (edit as needed)
magnification = 40
tile_size = 1024
overlap = 0.0

rfdetr_model = 'base'
rfdetr_epochs = 60
rfdetr_batch_size = 4
rfdetr_grad_accum = 2
rfdetr_lr = 5e-5
rfdetr_lr_scheduler = 'cosine'
rfdetr_lr_min_factor = 0.1
rfdetr_warmup_epochs = 2
rfdetr_weight_decay = 0.01
rfdetr_imgsz = 1008  # must be divisible by 56 for base/small/nano/large
rfdetr_workers = 6
rfdetr_early_stop = 8
rfdetr_drop_path = 0.1
rfdetr_aug = 'custom'

annotator_conf_thresh = 0.5
annotator_nms_iou = 0.5
annotator_magnification = 20
annotator_overlap = 0.10
annotator_compression = 'best_focal_plane'  # or 'focus_stack'
annotator_rfdetr_variant = 'base'

# ---- Imports from src ----
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data.ndpa_reader import NDPAData
from src.data.ndpi_reader import NDPIData
from src.preprocessing.generate_tiles import generate_tiles
from src.preprocessing.h5_utils import list_h5_paths
from src.preprocessing.postprocess_tiles import process_h5_file
from src.preprocessing.split_data import split_data as split_data_fn
from src.preprocessing.export_coco import ExportMode, run as export_coco_run
from src.annotator.config import AnnotatorConfig
from src.annotator.pipeline import NDPIAnnotator
from src.models.rfdetr.train import (
    _MODEL_CLASSES,
    _MODEL_IMGSZ_DIVISOR,
    AUG_CONFIG,
    _prepare_roboflow_layout,
    _read_class_names,
)

# Basic path checks
assert repo_root.is_dir(), f'Repo not found: {repo_root}'
assert input_ndpi_dir.is_dir(), f'NDPI dir not found: {input_ndpi_dir}'
assert annotation_map_csv.is_file(), f'CSV not found: {annotation_map_csv}'

## 1) Generate H5 tiles from NDPI+NDPA

In [ ]:
h5_dir.mkdir(parents=True, exist_ok=True)

annotation_map = pd.read_csv(annotation_map_csv)
specimen_to_category = {
    str(row['Specimen_name']).lower(): str(row['Category'])
    for _, row in annotation_map.iterrows()
}
categories = sorted(set(specimen_to_category.values()))
category_to_index = {cat: idx for idx, cat in enumerate(categories)}
specimen_to_index = {
    label: category_to_index[cat]
    for label, cat in specimen_to_category.items()
    if cat in category_to_index
}

ndpi_paths = sorted(input_ndpi_dir.glob('*.ndpi'))
if not ndpi_paths:
    raise FileNotFoundError(f'No .ndpi files found in {input_ndpi_dir}')

for ndpi_path in ndpi_paths:
    ndpa_path = ndpi_path.with_suffix(ndpi_path.suffix + '.ndpa')
    if not ndpa_path.is_file():
        print(f'Skipping (no NDPA): {ndpi_path.name}')
        continue
    ndpi = NDPIData(str(ndpi_path))
    ndpa = NDPAData(str(ndpa_path))
    generate_tiles(
        ndpi_data=ndpi,
        ndpa_data=ndpa,
        magnification=magnification,
        tile_size=tile_size,
        overlap=overlap,
        output_dir=str(h5_dir),
        label_map=specimen_to_index,
    )

## 2) Postprocess tiles (focus stack + rankings)

In [ ]:
h5_paths = list_h5_paths(str(h5_dir))
if not h5_paths:
    raise FileNotFoundError(f'No .h5 files found in {h5_dir}')

for h5_path in h5_paths:
    process_h5_file(
        h5_path=h5_path,
        write_focus_stacked=True,
        write_rankings=True,
        write_mip=False,
    )

## 3) Split slides into train/val/test

In [ ]:
splits_dir.mkdir(parents=True, exist_ok=True)
splits = split_data_fn(str(h5_dir), seed=67)

splits_json = splits_dir / 'train_val_test.json'
with open(splits_json, 'w') as f:
    json.dump(splits, f, indent=2)

print(f'Saved splits to {splits_json}')
print(f"Train: {len(splits['train'])}, Val: {len(splits['val'])}, Test: {len(splits['test'])}")

## 4) Export focus-stacked COCO (single class, filtered boxes)

In [ ]:
coco_dir.mkdir(parents=True, exist_ok=True)
mode = ExportMode(kind='focus_stack', plane_indices=[])
export_coco_run(
    h5_root=str(h5_dir),
    output_dir=str(coco_dir),
    mode=mode,
    splits_json=str(splits_json),
    workers=4,
    image_format='jpeg',
    single_cls=True,
    metadata_json=None,
    filter_bboxes=True,
 )

## 5) Train RF-DETR

In [ ]:
rfdetr_out.mkdir(parents=True, exist_ok=True)
divisor = _MODEL_IMGSZ_DIVISOR[rfdetr_model]
if rfdetr_imgsz % divisor != 0:
    raise ValueError(
        f'--imgsz {rfdetr_imgsz} is not divisible by {divisor} for model {rfdetr_model}.'
    )

staging_dir = rfdetr_out / '.rfdetr_dataset'
_prepare_roboflow_layout(str(coco_dir), str(staging_dir))
class_names = _read_class_names(str(coco_dir))
aug = dict(AUG_CONFIG[rfdetr_aug])

model = _MODEL_CLASSES[rfdetr_model]()
model.train(
    dataset_dir=str(staging_dir),
    epochs=rfdetr_epochs,
    batch_size=rfdetr_batch_size,
    grad_accum_steps=rfdetr_grad_accum,
    lr=rfdetr_lr,
    lr_scheduler=rfdetr_lr_scheduler,
    lr_min_factor=rfdetr_lr_min_factor,
    warmup_epochs=rfdetr_warmup_epochs,
    weight_decay=rfdetr_weight_decay,
    resolution=rfdetr_imgsz,
    output_dir=str(rfdetr_out),
    num_workers=rfdetr_workers,
    class_names=class_names,
    early_stopping=True,
    early_stopping_patience=rfdetr_early_stop,
    early_stopping_use_ema=True,
    aug_config=aug,
    drop_path=rfdetr_drop_path,
 )

## 6) Run annotator on a new slide

In [ ]:
# Update this to the best checkpoint produced in the run directory
checkpoint_path = rfdetr_out / 'checkpoint_best_ema.pth'
assert checkpoint_path.is_file(), f'Checkpoint not found: {checkpoint_path}'

annotator_out.mkdir(parents=True, exist_ok=True)
config = AnnotatorConfig(
    ndpi_path=str(annotate_ndpi_path),
    output_dir=str(annotator_out),
    model_name='rfdetr',
    checkpoint_path=str(checkpoint_path),
    overlap=annotator_overlap,
    magnification=annotator_magnification,
    confidence_threshold=annotator_conf_thresh,
    nms_iou_threshold=annotator_nms_iou,
    compression_method=annotator_compression,
    rfdetr_variant=annotator_rfdetr_variant,
 )
annotator = NDPIAnnotator(config)
detections = annotator.run()
print(f'Wrote {len(detections)} detections to {annotator_out}')